# GEANT h96 Colab migration v1

Preparation and preflight only. No upload or training is performed automatically. Official test-period data is excluded and remains sealed. Read `docs/geant_horizon96_colab_migration.md`; independently review and approve all four bundle/manifest SHA-256 values before transfer. Training requires separate authorization.

In [ ]:
# A-B: runtime facts and pinned environment verification; no project import or installation.
import os, platform, shutil, sys
import psutil
print({'python':sys.version,'platform':platform.platform(),'kernel':platform.release(),'cpu':platform.processor(),'logical_cpus':os.cpu_count(),'os_available_ram_bytes':psutil.virtual_memory().available,'disk_free_bytes':shutil.disk_usage('/content').free})
import numpy, torch
from importlib.metadata import version
env={'numpy':numpy.__version__,'torch':torch.__version__,'psutil':psutil.__version__,'pytest':version('pytest'),'cuda_available':torch.cuda.is_available(),'torch_cuda_build':torch.version.cuda,'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),'selected_device':'cpu'}
print(env)
assert tuple(sys.version_info[:2])==(3,12) and env['numpy']=='1.26.4' and env['torch'].split('+',1)[0]=='2.12.0' and env['psutil']=='7.2.2' and env['pytest']=='9.1.1'
print('A visible GPU is recorded but not selected; the migration policy remains CPU-only.')

In [ ]:
# C-D: explicit upload of the two archives and two sidecars only. No Drive mount or credentials.
from pathlib import Path
from google.colab import files
UPLOAD=Path('/content/colab_upload'); UPLOAD.mkdir(parents=True,exist_ok=False)
uploaded=files.upload()
expected={'geant_h96_colab_source_v1.zip','source_snapshot_manifest.json','geant_h96_colab_trainval_v1.zip','dataset_bundle_manifest.json'}
assert set(uploaded)==expected, f'Unexpected upload names: {sorted(uploaded)}'
for name,blob in uploaded.items():
    with (UPLOAD/name).open('xb') as f: f.write(blob)
print('Four expected files saved; no bundle contents have been extracted.')

In [ ]:
# E: enter values approved independently from the local sidecars; blank values intentionally stop.
EXPECTED_SOURCE_ZIP_SHA256=''
EXPECTED_SOURCE_MANIFEST_SHA256=''
EXPECTED_DATA_ZIP_SHA256=''
EXPECTED_DATA_MANIFEST_SHA256=''
assert all(len(x)==64 for x in (EXPECTED_SOURCE_ZIP_SHA256,EXPECTED_SOURCE_MANIFEST_SHA256,EXPECTED_DATA_ZIP_SHA256,EXPECTED_DATA_MANIFEST_SHA256)), 'Paste four independently reviewed digests first.'

In [ ]:
# E-G: stdlib-only hash/member checks, safe extraction, and exact inventory enforcement.
import hashlib, json, stat, zipfile
from pathlib import PurePosixPath
def sha(b): return hashlib.sha256(b).hexdigest()
def rel(n):
    p=PurePosixPath(n)
    assert n and '\\' not in n and not p.is_absolute() and all(x not in ('','.','..') for x in p.parts), n
    return p
def check_and_extract(zip_path, manifest_path, want_zip, want_manifest, kind, dest):
    mb=manifest_path.read_bytes(); assert sha(mb)==want_manifest, 'manifest digest mismatch'
    m=json.loads(mb); zb=zip_path.read_bytes(); assert sha(zb)==want_zip==m['archive_sha256'], 'archive digest mismatch'
    key='entries' if kind=='source' else 'artifacts'; entries=m[key]; names=[e['path'] for e in entries]
    assert len(names)==len({n.casefold() for n in names}), 'duplicate or case-conflicting manifest paths'
    allowed=set(names)|({'PORTABLE_SOURCE_MANIFEST.json'} if kind=='source' else set())
    with zipfile.ZipFile(zip_path) as z:
        infos=z.infolist(); got=[i.filename for i in infos]
        assert len(got)==len({n.casefold() for n in got}) and set(got)==allowed, 'archive inventory mismatch'
        for i in infos:
            rel(i.filename); mode=(i.external_attr>>16)&0xffff
            assert not i.is_dir() and not stat.S_ISLNK(mode), 'directory/symlink archive member'
        if kind=='source':
            inner=json.loads(z.read('PORTABLE_SOURCE_MANIFEST.json')); assert inner['schema']==m['schema'] and inner['entries']==entries
        for e in entries:
            b=z.read(e['path']); assert len(b)==e['size_bytes'] and sha(b)==e['sha256'], e['path']
        dest=Path(dest); assert not dest.exists(), f'existing destination: {dest}'
        dest.mkdir(parents=True,exist_ok=False); root=dest.resolve()
        for n in got:
            p=rel(n); out=dest.joinpath(*p.parts); out.parent.mkdir(parents=True,exist_ok=True)
            assert root in out.parent.resolve().parents or out.parent.resolve()==root, 'extraction escape'
            with out.open('xb') as f: f.write(z.read(n))
    return m
PROJECT=Path('/content/sdn_project')
SM=check_and_extract(UPLOAD/'geant_h96_colab_source_v1.zip',UPLOAD/'source_snapshot_manifest.json',EXPECTED_SOURCE_ZIP_SHA256,EXPECTED_SOURCE_MANIFEST_SHA256,'source',PROJECT)
DM=json.loads((UPLOAD/'dataset_bundle_manifest.json').read_text()); assert DM['archive_sha256']==EXPECTED_DATA_ZIP_SHA256
DATA_NAMES={'metadata.json','timestamps_train.npy','timestamps_validation.npy','utilization_train.npy','utilization_validation.npy','valid_train.npy','valid_validation.npy','window_index_train_h96.json','window_index_validation_h96.json'}
assert {e['path'] for e in DM['artifacts']}==DATA_NAMES and not any('test' in n.casefold() or 'h4' in n.casefold() for n in DATA_NAMES)
DATA=PROJECT/'data/processed/geant_horizon96_streaming_v1'
check_and_extract(UPLOAD/'geant_h96_colab_trainval_v1.zip',UPLOAD/'dataset_bundle_manifest.json',EXPECTED_DATA_ZIP_SHA256,EXPECTED_DATA_MANIFEST_SHA256,'data',DATA)
print('Only the approved source snapshot and nine train/validation members were extracted.')

In [ ]:
# H-J: isolated launcher process checks exact imports/data/topology/resources and runs synthetic tests, then dry-run.
import subprocess
LAUNCH=PROJECT/'phase9_geant/colab_h96_v1/launch.py'
base=[sys.executable,'-I','-S',str(LAUNCH),'--repository-root',str(PROJECT),'--source-bundle',str(UPLOAD/'geant_h96_colab_source_v1.zip'),'--source-manifest',str(UPLOAD/'source_snapshot_manifest.json'),'--expected-source-bundle-sha256',EXPECTED_SOURCE_ZIP_SHA256,'--expected-source-manifest-sha256',EXPECTED_SOURCE_MANIFEST_SHA256,'--dataset-bundle',str(UPLOAD/'geant_h96_colab_trainval_v1.zip'),'--dataset-manifest',str(UPLOAD/'dataset_bundle_manifest.json'),'--expected-dataset-bundle-sha256',EXPECTED_DATA_ZIP_SHA256,'--expected-dataset-manifest-sha256',EXPECTED_DATA_MANIFEST_SHA256]
TEST_TEMP=Path('/content/colab_pytest_temp'); assert not TEST_TEMP.exists(), f'Refusing existing pytest temp: {TEST_TEMP}'
tests=[sys.executable,'-m','pytest','-q',str(PROJECT/'phase9_geant/colab_h96_v1/tests/test_portability.py'),str(PROJECT/'phase9_geant/tests/test_horizon96_streaming_experiment_v2.py'),str(PROJECT/'phase9_geant/tests/test_horizon96_streaming_experiment.py'),str(PROJECT/'phase9_geant/tests/test_spatiotemporal_link.py'),'--basetemp',str(TEST_TEMP)]
assert subprocess.run(tests,cwd=PROJECT,check=False).returncode==0, 'Synthetic tests failed; stop.'
assert subprocess.run(base+['--dry-run'],cwd='/content',check=False).returncode==0, 'Isolated dry-run failed; stop.'

In [ ]:
# K-L: display the proposed first-run command; default state is deliberately no-training.
TRAIN_COMMAND=base+['--run-training','--authorize-training','--seed','42','--adjacency-mode','topology']
print('Proposed command:', ' '.join(TRAIN_COMMAND))
TRAINING_AUTHORIZED=False
if TRAINING_AUTHORIZED is True:
    completed=subprocess.run(TRAIN_COMMAND,cwd='/content',check=False)
    assert completed.returncode==0, f'Training stopped with exit {completed.returncode}; do not retry or resume.'
else:
    print('Training not authorized; no model was trained.')